# Data Preprocessing

This notebook checks empty rows/columns, duplicate rows/columns, constant columns, and flags outliers using train-derived IQR bounds. Raw `train` and `test` are kept unchanged.

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 160)

## Load Data

In [2]:
train = pd.read_csv("data/train_imputed.csv")
test = pd.read_csv("data/test_imputed.csv")

print("train shape:", train.shape)
print("test shape:", test.shape)

train shape: (690088, 15)
test shape: (295753, 14)


In [3]:
ID_COL = "id"
TARGET_COL = "health_condition"

train_feature_cols = [col for col in train.columns if col not in [ID_COL, TARGET_COL]]
test_feature_cols = [col for col in test.columns if col != ID_COL]

print("train feature columns:", train_feature_cols)
print("test feature columns:", test_feature_cols)

train feature columns: ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake', 'diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']
test feature columns: ['sleep_duration', 'heart_rate', 'bmi', 'calorie_expenditure', 'step_count', 'exercise_duration', 'water_intake', 'diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']


## Empty Columns And Rows

A row is treated as empty if every non-ID value is missing. This avoids `id` hiding an otherwise empty row.

In [4]:
train_empty_cols = train.columns[train.isna().all()].tolist()
test_empty_cols = test.columns[test.isna().all()].tolist()

train_empty_rows_mask = train[[col for col in train.columns if col != ID_COL]].isna().all(axis=1)
test_empty_rows_mask = test[[col for col in test.columns if col != ID_COL]].isna().all(axis=1)

empty_report = pd.DataFrame([
    {"dataset": "train", "empty_columns": train_empty_cols, "empty_row_count": int(train_empty_rows_mask.sum())},
    {"dataset": "test", "empty_columns": test_empty_cols, "empty_row_count": int(test_empty_rows_mask.sum())},
])

empty_report

,dataset,empty_columns,empty_row_count
0,train,[],0
1,test,[],0


## Duplicate Rows

Exact duplicate rows include `id`, so they are usually zero. Duplicate data rows ignore `id`, which is more useful for model training.

In [5]:
train_exact_duplicate_rows = int(train.duplicated().sum())
test_exact_duplicate_rows = int(test.duplicated().sum())

train_duplicate_data_rows_mask = train.duplicated(subset=[col for col in train.columns if col != ID_COL], keep="first")
test_duplicate_data_rows_mask = test.duplicated(subset=[col for col in test.columns if col != ID_COL], keep="first")

duplicate_row_report = pd.DataFrame([
    {
        "dataset": "train",
        "exact_duplicate_rows": train_exact_duplicate_rows,
        "duplicate_rows_ignoring_id": int(train_duplicate_data_rows_mask.sum()),
    },
    {
        "dataset": "test",
        "exact_duplicate_rows": test_exact_duplicate_rows,
        "duplicate_rows_ignoring_id": int(test_duplicate_data_rows_mask.sum()),
    },
])

duplicate_row_report

,dataset,exact_duplicate_rows,duplicate_rows_ignoring_id
0,train,0,0
1,test,0,0


## Duplicate Columns

In [6]:
def find_duplicate_columns(df):
    duplicate_groups = []
    used = set()

    for i, col in enumerate(df.columns):
        if col in used:
            continue

        group = [col]
        for other_col in df.columns[i + 1:]:
            if other_col in used:
                continue
            if df[col].equals(df[other_col]):
                group.append(other_col)

        if len(group) > 1:
            duplicate_groups.append(group)
            used.update(group)

    return duplicate_groups


def columns_to_drop_from_duplicate_groups(duplicate_groups, protected_cols):
    drops = []
    for group in duplicate_groups:
        keep = next((col for col in group if col in protected_cols), group[0])
        drops.extend([col for col in group if col != keep])
    return drops

train_duplicate_column_groups = find_duplicate_columns(train)
test_duplicate_column_groups = find_duplicate_columns(test)

duplicate_column_report = pd.DataFrame([
    {"dataset": "train", "duplicate_column_groups": train_duplicate_column_groups},
    {"dataset": "test", "duplicate_column_groups": test_duplicate_column_groups},
])

duplicate_column_report

,dataset,duplicate_column_groups
0,train,[]
1,test,[]


## Constant Columns

Constant columns have zero or one unique non-null value, so they do not add useful model signal.

In [7]:
train_constant_cols = [
    col for col in train.columns
    if col not in [ID_COL, TARGET_COL] and train[col].nunique(dropna=True) <= 1
]
test_constant_cols = [
    col for col in test.columns
    if col != ID_COL and test[col].nunique(dropna=True) <= 1
]

constant_column_report = pd.DataFrame([
    {"dataset": "train", "constant_columns": train_constant_cols},
    {"dataset": "test", "constant_columns": test_constant_cols},
])

constant_column_report

,dataset,constant_columns
0,train,[]
1,test,[]


## Apply Basic Cleaning

Column removal rules are learned from train and then applied to both train and test when the column exists. Row removals are applied only to train because test IDs normally must be preserved for submission.

In [8]:
protected_cols = {ID_COL, TARGET_COL}
train_duplicate_cols_to_drop = columns_to_drop_from_duplicate_groups(
    train_duplicate_column_groups,
    protected_cols=protected_cols,
)

columns_to_drop_from_train_rules = sorted(set(
    col for col in train_empty_cols + train_duplicate_cols_to_drop + train_constant_cols
    if col not in protected_cols
))

train_preprocessed = train.copy()
test_preprocessed = test.copy()

train_preprocessed = train_preprocessed.drop(columns=[
    col for col in columns_to_drop_from_train_rules
    if col in train_preprocessed.columns
])
test_preprocessed = test_preprocessed.drop(columns=[
    col for col in columns_to_drop_from_train_rules
    if col in test_preprocessed.columns
])

train_preprocessed = train_preprocessed.loc[~train_empty_rows_mask].copy()
train_preprocessed = train_preprocessed.loc[~train_duplicate_data_rows_mask].copy()

basic_cleaning_report = pd.DataFrame([
    {"step": "drop train-derived empty/duplicate/constant columns", "removed": columns_to_drop_from_train_rules},
    {"step": "drop empty train rows", "removed": int(train_empty_rows_mask.sum())},
    {"step": "drop duplicate train rows ignoring id", "removed": int(train_duplicate_data_rows_mask.sum())},
    {"step": "keep test rows", "removed": 0},
])

print("train shape before basic cleaning:", train.shape)
print("train shape after basic cleaning:", train_preprocessed.shape)
print("test shape before basic cleaning:", test.shape)
print("test shape after basic cleaning:", test_preprocessed.shape)

basic_cleaning_report

train shape before basic cleaning: (690088, 15)
train shape after basic cleaning: (690088, 15)
test shape before basic cleaning: (295753, 14)
test shape after basic cleaning: (295753, 14)


,step,removed
0,drop train-derived empty/duplicate/constant columns,[]
1,drop empty train rows,0
2,drop duplicate train rows ignoring id,0
3,keep test rows,0


## IQR Outlier Flags

IQR bounds are learned from train only. Outlier rows are not removed by default because extreme health values may be useful signal. Instead, the notebook adds outlier flag features to both `train_preprocessed` and `test_preprocessed` using the same train-derived bounds.

In [9]:
numeric_feature_cols = [
    col for col in train_preprocessed.columns
    if col not in [ID_COL, TARGET_COL] and pd.api.types.is_numeric_dtype(train_preprocessed[col])
]

IQR_MULTIPLIER = 1.5

iqr_bounds_rows = []
for col in numeric_feature_cols:
    q1 = train_preprocessed[col].quantile(0.25)
    q3 = train_preprocessed[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - IQR_MULTIPLIER * iqr
    upper = q3 + IQR_MULTIPLIER * iqr
    iqr_bounds_rows.append({
        "column": col,
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "lower_bound": lower,
        "upper_bound": upper,
    })

iqr_bounds = pd.DataFrame(iqr_bounds_rows)
iqr_bounds

,column,q1,q3,iqr,lower_bound,upper_bound
0,sleep_duration,6.30,7.68,1.38,4.230,9.750
1,heart_rate,69.50,80.60,11.10,52.850,97.250
2,bmi,21.38,24.63,3.25,16.505,29.505
3,calorie_expenditure,2068.00,2438.00,370.00,1513.000,2993.000
4,step_count,5460.00,12089.00,6629.00,-4483.500,22032.500
5,exercise_duration,29.30,49.30,20.00,-0.700,79.300
6,water_intake,1.87,2.47,0.60,0.970,3.370


In [10]:
train_outlier_mask = pd.Series(False, index=train_preprocessed.index)
test_outlier_mask = pd.Series(False, index=test_preprocessed.index)
outlier_report_rows = []
outlier_flag_cols = []

for row in iqr_bounds.itertuples(index=False):
    flag_col = f"{row.column}_iqr_outlier"
    outlier_flag_cols.append(flag_col)

    train_col_outliers = (
        train_preprocessed[row.column].notna()
        & ((train_preprocessed[row.column] < row.lower_bound) | (train_preprocessed[row.column] > row.upper_bound))
    )
    train_preprocessed[flag_col] = train_col_outliers.astype(int)
    train_outlier_mask |= train_col_outliers

    if row.column in test_preprocessed.columns:
        test_col_outliers = (
            test_preprocessed[row.column].notna()
            & ((test_preprocessed[row.column] < row.lower_bound) | (test_preprocessed[row.column] > row.upper_bound))
        )
        test_preprocessed[flag_col] = test_col_outliers.astype(int)
        test_outlier_mask |= test_col_outliers
        test_outlier_count = int(test_col_outliers.sum())
    else:
        test_outlier_count = 0

    outlier_report_rows.append({
        "column": row.column,
        "flag_column": flag_col,
        "train_outlier_count": int(train_col_outliers.sum()),
        "train_outlier_pct": train_col_outliers.mean() * 100,
        "test_outlier_count_using_train_bounds": test_outlier_count,
        "test_outlier_pct_using_train_bounds": test_outlier_count / len(test_preprocessed) * 100 if len(test_preprocessed) else 0,
    })

train_preprocessed["any_iqr_outlier"] = train_outlier_mask.astype(int)
test_preprocessed["any_iqr_outlier"] = test_outlier_mask.astype(int)
outlier_flag_cols.append("any_iqr_outlier")

outlier_report = pd.DataFrame(outlier_report_rows).sort_values(
    "train_outlier_count",
    ascending=False,
)

outlier_report.style.format({
    "train_outlier_pct": "{:.2f}",
    "test_outlier_pct_using_train_bounds": "{:.2f}",
})

,column,flag_column,train_outlier_count,train_outlier_pct,test_outlier_count_using_train_bounds,test_outlier_pct_using_train_bounds
3,calorie_expenditure,calorie_expenditure_iqr_outlier,20574,2.98,8841,2.99
6,water_intake,water_intake_iqr_outlier,20026,2.90,8696,2.94
0,sleep_duration,sleep_duration_iqr_outlier,16416,2.38,7104,2.40
2,bmi,bmi_iqr_outlier,6291,0.91,2623,0.89
1,heart_rate,heart_rate_iqr_outlier,3441,0.50,1183,0.40
5,exercise_duration,exercise_duration_iqr_outlier,115,0.02,33,0.01
4,step_count,step_count_iqr_outlier,0,0.00,0,0.00


In [11]:
print("train rows flagged by IQR, not removed:", int(train_outlier_mask.sum()))
print("test rows flagged by train IQR bounds, not removed:", int(test_outlier_mask.sum()))
print("outlier flag columns added:", outlier_flag_cols)

print("final train_preprocessed shape:", train_preprocessed.shape)
print("final test_preprocessed shape:", test_preprocessed.shape)

train rows flagged by IQR, not removed: 64389
test rows flagged by train IQR bounds, not removed: 27519
outlier flag columns added: ['sleep_duration_iqr_outlier', 'heart_rate_iqr_outlier', 'bmi_iqr_outlier', 'calorie_expenditure_iqr_outlier', 'step_count_iqr_outlier', 'exercise_duration_iqr_outlier', 'water_intake_iqr_outlier', 'any_iqr_outlier']
final train_preprocessed shape: (690088, 23)
final test_preprocessed shape: (295753, 22)


## Final Checks

In [12]:
final_check = pd.DataFrame([
    {
        "dataset": "train_preprocessed",
        "rows": len(train_preprocessed),
        "columns": train_preprocessed.shape[1],
        "empty_rows_non_id": int(train_preprocessed[[col for col in train_preprocessed.columns if col != ID_COL]].isna().all(axis=1).sum()),
        "exact_duplicate_rows": int(train_preprocessed.duplicated().sum()),
        "duplicate_rows_ignoring_id": int(train_preprocessed.duplicated(subset=[col for col in train_preprocessed.columns if col != ID_COL], keep="first").sum()),
    },
    {
        "dataset": "test_preprocessed",
        "rows": len(test_preprocessed),
        "columns": test_preprocessed.shape[1],
        "empty_rows_non_id": int(test_preprocessed[[col for col in test_preprocessed.columns if col != ID_COL]].isna().all(axis=1).sum()),
        "exact_duplicate_rows": int(test_preprocessed.duplicated().sum()),
        "duplicate_rows_ignoring_id": int(test_preprocessed.duplicated(subset=[col for col in test_preprocessed.columns if col != ID_COL], keep="first").sum()),
    },
])

final_check

,dataset,rows,columns,empty_rows_non_id,exact_duplicate_rows,duplicate_rows_ignoring_id
0,train_preprocessed,690088,23,0,0,0
1,test_preprocessed,295753,22,0,0,0


In [13]:
train_preprocessed.head()

,id,health_condition,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender,sleep_duration_iqr_outlier,heart_rate_iqr_outlier,bmi_iqr_outlier,calorie_expenditure_iqr_outlier,step_count_iqr_outlier,exercise_duration_iqr_outlier,water_intake_iqr_outlier,any_iqr_outlier
0,0,unhealthy,5.22,70.6,25.66,2174.0,1326.0,19.8,1.86,veg,high,average,sedentary,yes,female,0,0,0,0,0,0,0,0
1,1,at-risk,5.53,71.3,25.84,1966.0,9891.0,49.9,1.26,non-veg,low,average,moderate,yes,other,0,0,0,0,0,0,0,0
2,2,unhealthy,5.29,75.4,24.54,2688.0,14216.0,38.1,1.60,veg,high,poor,active,yes,male,0,0,0,0,0,0,0,0
3,3,unhealthy,4.70,77.2,23.13,2630.0,7174.0,59.9,2.02,veg,high,average,active,occasional,female,0,0,0,0,0,0,0,0
4,4,at-risk,7.23,73.4,28.44,2560.0,6584.0,46.0,2.25,veg,medium,average,sedentary,yes,male,0,0,0,0,0,0,0,0


In [14]:
test_preprocessed.head()

,id,sleep_duration,heart_rate,bmi,calorie_expenditure,step_count,exercise_duration,water_intake,diet_type,stress_level,sleep_quality,physical_activity_level,smoking_alcohol,gender,sleep_duration_iqr_outlier,heart_rate_iqr_outlier,bmi_iqr_outlier,calorie_expenditure_iqr_outlier,step_count_iqr_outlier,exercise_duration_iqr_outlier,water_intake_iqr_outlier,any_iqr_outlier
0,690088,5.35,64.9,23.48,2745.0,14167.0,59.5,1.86,veg,high,poor,active,occasional,male,0,0,0,0,0,0,0,0
1,690089,6.99,83.1,22.42,1773.0,6801.0,24.5,2.40,balanced,high,poor,sedentary,yes,other,0,0,0,0,0,0,0,0
2,690090,6.68,59.7,24.14,3040.0,13250.0,48.5,2.76,balanced,medium,poor,active,no,male,0,0,0,1,0,0,0,1
3,690091,7.13,78.5,26.26,2494.0,6331.0,56.9,2.34,veg,low,good,moderate,yes,other,0,0,0,0,0,0,0,0
4,690092,5.49,77.7,23.29,1828.0,13894.0,39.4,2.45,veg,high,average,active,occasional,other,0,0,0,0,0,0,0,0


In [15]:
train_preprocessed.to_csv("data/train_preprocessed.csv", index=False)

In [16]:
test_preprocessed.to_csv("data/test_preprocessed.csv", index=False)